# Backend Specifications Deep Dive

qb-compiler ships with detailed specifications for 9 quantum backends across 5 vendors.
This notebook explores every backend's hardware characteristics and shows how to use them
for informed backend selection.

Topics covered:
- `BACKEND_CONFIGS` — all 9 backends and their specs
- `BackendSpec` fields and what they mean
- `max_circuit_depth_heuristic` — conservative depth limits
- `BackendTarget` — topology queries (connectivity, distance, neighbours)
- Cross-vendor comparisons: IBM Heron vs Rigetti vs IonQ vs IQM vs Quantinuum
- Building a backend selection decision tree

In [ ]:
from qb_compiler.config import BACKEND_CONFIGS, BackendSpec, get_backend_spec
from qb_compiler.backends.base import BackendTarget

## 1. BACKEND_CONFIGS overview

Every supported backend has a `BackendSpec` entry with hardware-specific metadata.
These specs drive viability checks, cost estimation, compilation strategy selection,
and depth limit heuristics.

### BackendSpec fields

| Field | Type | Description |
|-------|------|-------------|
| `provider` | `str` | Vendor name (`ibm`, `rigetti`, `ionq`, `iqm`, `quantinuum`) |
| `n_qubits` | `int` | Total physical qubits |
| `basis_gates` | `tuple[str, ...]` | Native gate set |
| `coupling_map` | `str` | Human-readable topology description |
| `cost_per_shot` | `float` | Cost in USD per shot |
| `median_cx_error` | `float` | Median two-qubit gate error rate |
| `median_readout_error` | `float` | Median measurement error rate |
| `t1_us` | `float` | Median T1 relaxation time in microseconds |
| `t2_us` | `float` | Median T2 dephasing time in microseconds |

In [ ]:
print(f"Total backends configured: {len(BACKEND_CONFIGS)}\n")

header = f"{'Backend':>16s} | {'Provider':>10s} | {'Qubits':>6s} | {'CX Err':>8s} | {'RO Err':>8s} | {'$/shot':>10s}"
sep = "-" * len(header)
print(sep)
print(header)
print(sep)

for name, spec in sorted(BACKEND_CONFIGS.items()):
    print(
        f"{name:>16s} | {spec.provider:>10s} | {spec.n_qubits:>6d} | "
        f"{spec.median_cx_error:>8.4f} | {spec.median_readout_error:>8.4f} | "
        f"${spec.cost_per_shot:>9.5f}"
    )
print(sep)

In [ ]:
# Inspect a single backend in detail
spec = get_backend_spec("ibm_fez")

print(f"Backend: ibm_fez")
print(f"  Provider:        {spec.provider}")
print(f"  Qubits:          {spec.n_qubits}")
print(f"  Basis gates:     {spec.basis_gates}")
print(f"  Coupling map:    {spec.coupling_map}")
print(f"  Cost per shot:   ${spec.cost_per_shot:.5f}")
print(f"  Median CX error: {spec.median_cx_error}")
print(f"  Median RO error: {spec.median_readout_error}")
print(f"  T1:              {spec.t1_us} us")
print(f"  T2:              {spec.t2_us} us")

## 2. max_circuit_depth_heuristic

Each `BackendSpec` computes a conservative upper bound on useful circuit depth.
Beyond this depth, decoherence dominates and circuit output is indistinguishable from noise.

The heuristic uses `T2 / (gate_time * 2)` where gate time is ~400 ns for
superconducting qubits and ~200 us for trapped ions. Trapped-ion backends have
enormously longer coherence times, but their gate times are also much longer.

In [ ]:
print(f"{'Backend':>16s} | {'Max Depth':>10s} | {'T2 (us)':>10s} | {'Technology':>15s}")
print("-" * 60)

for name, spec in sorted(BACKEND_CONFIGS.items()):
    tech = "trapped-ion" if spec.provider in ("ionq", "quantinuum") else "superconducting"
    print(
        f"{name:>16s} | {spec.max_circuit_depth_heuristic:>10,d} | "
        f"{spec.t2_us:>10.0f} | {tech:>15s}"
    )

## 3. Gate set differences

Each vendor uses a different native gate set. The compiler's basis translation pass
decomposes your circuit into the target backend's native gates.

In [ ]:
# Unique gate sets across all backends
seen_sets = {}
for name, spec in sorted(BACKEND_CONFIGS.items()):
    key = spec.basis_gates
    if key not in seen_sets:
        seen_sets[key] = []
    seen_sets[key].append(name)

print("Distinct native gate sets:\n")
for gates, backends in seen_sets.items():
    print(f"  {', '.join(gates)}")
    for b in backends:
        print(f"    -> {b}")
    print()

# What each gate set means in practice
explanations = {
    "IBM": "CX (CNOT) as native 2Q gate, SX/RZ for 1Q. Heavy-hex topology = limited connectivity.",
    "Rigetti": "CZ as native 2Q gate, RX/RZ for 1Q. Octagonal lattice = moderate connectivity.",
    "IonQ": "Molmer-Sorensen (MS) as native 2Q gate, GPI/GPI2 for 1Q. All-to-all (no routing).",
    "IQM": "CZ as native 2Q gate, PRX for 1Q. Square or star topology.",
    "Quantinuum": "ZZ as native 2Q gate, RZ/U1Q for 1Q. All-to-all via ion shuttling.",
}

print("Gate set implications:")
for provider, explanation in explanations.items():
    print(f"  {provider:12s}: {explanation}")

## 4. BackendTarget — topology queries

`BackendTarget` wraps a backend's physical constraints and provides topology query
methods. Key methods:

| Method | Returns | Description |
|--------|---------|-------------|
| `supports_gate(name)` | `bool` | Whether `name` is in the native gate set |
| `is_fully_connected` | `bool` | True if coupling_map is empty (all-to-all) |
| `are_connected(q1, q2)` | `bool` | Whether q1 and q2 share a direct coupling |
| `qubit_distance(q1, q2)` | `int` | Shortest path hops between q1 and q2 |
| `neighbours(q)` | `frozenset[int]` | Qubits directly coupled to q |

In [ ]:
# Build a BackendTarget for a small heavy-hex-like topology
heavy_hex_edges = [
    (0, 1), (1, 2), (2, 3), (3, 4),
    (0, 5), (2, 6), (4, 7),
    (5, 6), (6, 7),
]
# Add reverse edges (undirected)
coupling = heavy_hex_edges + [(b, a) for a, b in heavy_hex_edges]

ibm_target = BackendTarget(
    n_qubits=8,
    basis_gates=("cx", "rz", "sx", "x"),
    coupling_map=coupling,
    name="mini_heavy_hex",
)

print(ibm_target)
print(f"Fully connected: {ibm_target.is_fully_connected}")
print()

# supports_gate()
print(f"Supports CX: {ibm_target.supports_gate('cx')}")
print(f"Supports CZ: {ibm_target.supports_gate('cz')}")
print(f"Supports MS: {ibm_target.supports_gate('ms')}")

In [ ]:
# are_connected() and qubit_distance()
print("Direct connections:")
print(f"  Q0 -- Q1: {ibm_target.are_connected(0, 1)}")
print(f"  Q0 -- Q2: {ibm_target.are_connected(0, 2)}")
print(f"  Q0 -- Q5: {ibm_target.are_connected(0, 5)}")
print()

# Shortest path distances from Q0
print("Shortest path distances from Q0:")
for q in range(ibm_target.n_qubits):
    dist = ibm_target.qubit_distance(0, q)
    print(f"  Q0 -> Q{q}: {dist} hop{'s' if dist != 1 else ''}")
print()

# neighbours()
print("Neighbour map:")
for q in range(ibm_target.n_qubits):
    nbrs = sorted(ibm_target.neighbours(q))
    print(f"  Q{q} neighbours: {nbrs} (degree {len(nbrs)})")

In [ ]:
# All-to-all connectivity (IonQ / Quantinuum)
ionq_target = BackendTarget(
    n_qubits=5,
    basis_gates=("gpi", "gpi2", "ms"),
    coupling_map=[],  # empty = all-to-all
    name="ionq_5q",
)

print(f"IonQ target: {ionq_target}")
print(f"Fully connected: {ionq_target.is_fully_connected}")
print(f"Q0 -- Q4 connected: {ionq_target.are_connected(0, 4)}")
print(f"Q0 -- Q4 distance:  {ionq_target.qubit_distance(0, 4)}")
print(f"Q2 neighbours: {sorted(ionq_target.neighbours(2))}")

## 5. Connectivity topology comparison

| Provider | Topology | Routing overhead | Best for |
|----------|----------|-----------------|----------|
| **IBM** | Heavy-hex | High (many SWAPs needed) | Large qubit count, moderate depth |
| **Rigetti** | Octagonal lattice | Moderate | Medium circuits |
| **IonQ** | All-to-all | None (no routing) | Deep circuits, small qubit count |
| **IQM** | Square/Star | Moderate to high | Small-medium circuits |
| **Quantinuum** | All-to-all | None (no routing) | Highest fidelity, small circuits |

## 6. Qubit quality rankings

In [ ]:
# Rank by 2Q gate fidelity (lower error = better)
by_cx = sorted(BACKEND_CONFIGS.items(), key=lambda x: x[1].median_cx_error)

print("Ranked by 2Q gate fidelity (best first):")
for i, (name, spec) in enumerate(by_cx, 1):
    fidelity = 1 - spec.median_cx_error
    print(f"  {i}. {name:>16s}  error={spec.median_cx_error:.4f}  fidelity={fidelity:.4f}")

print()

# Rank by readout fidelity
by_ro = sorted(BACKEND_CONFIGS.items(), key=lambda x: x[1].median_readout_error)

print("Ranked by readout fidelity (best first):")
for i, (name, spec) in enumerate(by_ro, 1):
    print(f"  {i}. {name:>16s}  error={spec.median_readout_error:.4f}")

In [ ]:
# Rank by cost efficiency (fidelity per dollar per shot)
print("Ranked by cost efficiency (2Q gate fidelity per dollar):")
by_value = sorted(
    BACKEND_CONFIGS.items(),
    key=lambda x: (1 - x[1].median_cx_error) / x[1].cost_per_shot,
    reverse=True,
)

for i, (name, spec) in enumerate(by_value, 1):
    fidelity = 1 - spec.median_cx_error
    fpd = fidelity / spec.cost_per_shot
    print(f"  {i}. {name:>16s}  fidelity/$ = {fpd:>10,.0f}  (${spec.cost_per_shot:.5f}/shot)")

## 7. Which backend for which workload?

Different circuit characteristics favour different backends. The function below
implements a simple decision tree that filters by qubit count, budget, connectivity,
and depth limits, then ranks by estimated fidelity.

In [ ]:
def recommend_backend(
    n_qubits: int,
    circuit_depth: int,
    needs_all_to_all: bool = False,
    max_budget_per_shot: float = 1.0,
) -> list[tuple[str, str]]:
    """Simple backend selection decision tree.

    Returns a list of (backend, reason) tuples, best first.
    """
    candidates = []

    for name, spec in BACKEND_CONFIGS.items():
        if spec.n_qubits < n_qubits:
            continue
        if spec.cost_per_shot > max_budget_per_shot:
            continue
        is_all_to_all = spec.provider in ("ionq", "quantinuum")
        if needs_all_to_all and not is_all_to_all:
            continue
        if circuit_depth > spec.max_circuit_depth_heuristic:
            continue

        fidelity = (1 - spec.median_cx_error) ** circuit_depth * (1 - spec.median_readout_error)
        reason = (
            f"fidelity~{fidelity:.3f}, {spec.n_qubits}q, "
            f"${spec.cost_per_shot:.5f}/shot"
        )
        candidates.append((name, reason, fidelity))

    candidates.sort(key=lambda x: -x[2])
    return [(name, reason) for name, reason, _ in candidates]


# Scenario 1: Small QAOA circuit
print("Scenario: 5-qubit QAOA, depth 20")
for name, reason in recommend_backend(5, 20):
    print(f"  {name:>16s}: {reason}")
print()

# Scenario 2: Large circuit needing many qubits
print("Scenario: 50-qubit circuit, depth 100, budget $0.01/shot")
results = recommend_backend(50, 100, max_budget_per_shot=0.01)
if results:
    for name, reason in results:
        print(f"  {name:>16s}: {reason}")
else:
    print("  No backends match these constraints.")
print()

# Scenario 3: Deep circuit needing all-to-all
print("Scenario: 10-qubit deep circuit, depth 200, needs all-to-all")
results = recommend_backend(10, 200, needs_all_to_all=True)
if results:
    for name, reason in results:
        print(f"  {name:>16s}: {reason}")
else:
    print("  No backends match these constraints.")

## 8. Cost comparison at scale

The cost difference between vendors spans 5 orders of magnitude. This matters when
running large experiments.

In [ ]:
from qb_compiler.cost.estimator import CostEstimator

estimator = CostEstimator()

shot_counts = [1_000, 10_000, 100_000]

print(f"{'Backend':>16s}", end="")
for s in shot_counts:
    print(f" | {s:>10,d} shots", end="")
print()
print("-" * 60)

for name in sorted(BACKEND_CONFIGS.keys()):
    print(f"{name:>16s}", end="")
    for shots in shot_counts:
        est = estimator.estimate(name, shots)
        print(f" | ${est.total_cost_usd:>10,.2f}", end="")
    print()

## Summary: choosing the right backend

| Priority | Best choice | Why |
|----------|-------------|-----|
| Highest fidelity | `quantinuum_h2` | Lowest gate + readout error |
| Largest scale | `ibm_fez`, `ibm_marrakesh` | 156 qubits |
| Best value | `ibm_torino`, `ibm_fez` | Low per-shot cost with decent fidelity |
| All-to-all needed | `ionq_aria`, `ionq_forte`, `quantinuum_h2` | No routing overhead |
| Budget-constrained | IBM backends, `iqm_emerald` | Cheapest per shot |
| Deep circuits | Trapped-ion backends | Long coherence times |

Use `qb_compiler.recommender.BackendRecommender` for automated backend selection
that accounts for your specific circuit structure and calibration data.